In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from kneed import KneeLocator
from sklearn import metrics
from sklearn.cluster import KMeans

In [ ]:
data_file = Path('..') / 'data' / 'data_processed' / 'SUPERSTORE_MODELAGEM.csv'
df = pd.read_csv(data_file, sep=',')

In [ ]:
df.head()

In [ ]:
df_clustering = df[['ORDER_DATE', 'SHIP_MODE', 'CUSTOMER_ID', 'SEGMENT', 'CATEGORY', 'REGION', 'NET_SALES', 'PROFIT', 'NET_SALES_SCALED', 'PROFIT_SCALED']].copy()
df_clustering = df_clustering.reset_index(drop=True)

df_clustering = df_clustering.groupby('CUSTOMER_ID', as_index=False).agg({
    'ORDER_DATE': 'first',
    'SHIP_MODE': 'first',
    'CATEGORY': 'first',
    'REGION': 'first',
    'SEGMENT': 'first',
    'NET_SALES': 'sum',
    'PROFIT': 'sum',
    'NET_SALES_SCALED': 'sum',
    'PROFIT_SCALED': 'sum'
})

X_scaled = df_clustering.drop(columns=['CUSTOMER_ID', 'NET_SALES', 'PROFIT'])
X_scaled['ORDER_DATE'] = pd.to_datetime(df_clustering['ORDER_DATE'], errors='coerce').dt.year

In [ ]:
df_clustering.head()

In [ ]:
cols_one_hot = ['SHIP_MODE', 'SEGMENT', 'REGION', 'CATEGORY', 'ORDER_DATE']

for col in cols_one_hot:
    print(df_clustering[col].value_counts(), '\n')

In [ ]:
X_scaled = pd.get_dummies(
    X_scaled,
    columns=cols_one_hot)

In [ ]:
k_range = range(2, 8)

results = {
    'k': [],
    'inertia': [],
    'silhouette': [],
    'calinski_harabasz': [],
    'davies_bouldin': [],
}

for k in k_range:
    model = KMeans(n_clusters=k, 
        n_init=10,
        random_state=0,
        max_iter=200
    )
    labels = model.fit_predict(X_scaled)

    results['k'].append(k)
    results['inertia'].append(model.inertia_)
    results['silhouette'].append(
        metrics.silhouette_score(X_scaled, labels)
    )
    results['calinski_harabasz'].append(
        metrics.calinski_harabasz_score(X_scaled, labels)
    )
    results['davies_bouldin'].append(
        metrics.davies_bouldin_score(X_scaled, labels)
    )


df_metrics = pd.DataFrame(results).set_index('k')

knee = KneeLocator(
    x=df_metrics.index,
    y=df_metrics['inertia'],
    curve='convex',
    direction='decreasing'
)

k_optimal = knee.knee
if k_optimal is None:
    print('Elbow não encontrado')
    
k_sil = df_metrics['silhouette'].idxmax()
k_ch  = df_metrics['calinski_harabasz'].idxmax()
k_db  = df_metrics['davies_bouldin'].idxmin()

k_summary = pd.DataFrame({
    'method': ['silhouette', 'calinski_harabasz', 'davies_bouldin', 'elbow'],
    'k': [k_sil, k_ch, k_db, k_optimal]
})

k_summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

df_metrics['inertia'].plot(ax=axes[0, 0], title='Inertia')

if k_optimal is not None:
    axes[0, 0].axvline(k_optimal, linestyle='--')

df_metrics['silhouette'].plot(ax=axes[0, 1], title='Silhouette')
df_metrics['calinski_harabasz'].plot(ax=axes[1, 0], title='Calinski-Harabasz')
df_metrics['davies_bouldin'].plot(ax=axes[1, 1], title='Davies-Bouldin')

plt.tight_layout()
plt.show()

In [ ]:
kmeans_publico = KMeans(n_clusters=3, 
        n_init=10,
        random_state=0
    )
kmeans_publico.fit(X_scaled)
labels = kmeans_publico.labels_

df_clustering['CLUSTER'] = labels


In [ ]:
centroids = pd.DataFrame(
    kmeans_publico.cluster_centers_,
    columns=X_scaled.columns
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(centroids.round(2))

In [ ]:
resumo_clusters = (
    df_clustering
    .groupby('CLUSTER')
    .agg(
        qtd_clientes=('CUSTOMER_ID', 'count'),
        media_net_sales=('NET_SALES', 'mean'),
        media_profit=('PROFIT', 'mean')
    )
    .reset_index()
    .sort_values('CLUSTER')
    .reset_index(drop=True)
)

resumo_clusters.columns = [col.upper() for col in resumo_clusters.columns]

display(resumo_clusters.round(2))

In [ ]:
px.scatter(
    df_clustering,
    x='PROFIT',
    y='NET_SALES',
    color=df_clustering['CLUSTER'].astype(str)
)

In [ ]:
px.histogram(
    df_clustering,
    x='ORDER_DATE',
    y='PROFIT',
    nbins=(2022-2019)*12,
    color=df_clustering['CLUSTER'].astype(str)
)

In [ ]:
px.histogram(
    df_clustering,
    x='ORDER_DATE',
    y='NET_SALES',
    nbins=(2022-2019)*12,
    color=df_clustering['CLUSTER'].astype(str)
)